# Simple: Data Attribute Profiler with PAMOLA.CORE

**Goal**: Learn attribute profiling basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze attribute roles (identifiers, quasi-identifiers, sensitive)
- Compare entropy and uniqueness metrics
- Understand privacy implications

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_data_attribute_profiler_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.attribute import (
        DataAttributeProfilerOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from examples/data_examples/sample.csv
- Auto-creates sample data if file doesn't exist
- Review dataset structure and statistics

**What you'll see:**
- File path confirmation or creation message
- Dataset summary (50 records, 16 columns)
- First 5 records preview
- Column statistics (types, unique values, ranges)
- Mix of identifiers, demographics, and sensitive attributes

**Note:** Sample includes various attribute types: identifiers (patient_id, email), quasi-identifiers (birth_date, gender, region), and sensitive data (diagnosis, salary, ethnicity)

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample patient data with various attribute types
    sample_data = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, 51)],
        'email': [f'patient{i}@hospital.com' for i in range(1, 51)],
        'first_name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'] * 10,
        'last_name': ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones'] * 10,
        'birth_date': pd.date_range('1950-01-01', periods=50, freq='180D'),
        'gender': ['F', 'M', 'F', 'M', 'F'] * 10,
        'region': ['North', 'South', 'East', 'West', 'Central'] * 10,
        'city': ['Springfield', 'Riverside', 'Franklin', 'Clinton', 'Madison'] * 10,
        'postal_code': ['12345', '23456', '34567', '45678', '56789'] * 10,
        'diagnosis': ['Diabetes', 'Hypertension', 'Asthma', 'Arthritis', 'Migraine'] * 10,
        'treatment': ['Medication A', 'Medication B', 'Therapy C', 'Surgery D', 'Treatment E'] * 10,
        'salary': np.random.randint(30000, 120000, 50),
        'ethnicity': ['Asian', 'Caucasian', 'African', 'Hispanic', 'Other'] * 10,
        'notes': ['Patient shows good progress.' + ' ' * i for i in range(50)],
        'visit_count': np.random.randint(1, 20, 50),
        'created_at': pd.date_range('2024-01-01', periods=50, freq='D')
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Run to setup analysis environment
- No field customization needed (analyzes ALL columns automatically)
- Creates task directory and initializes tracking components

**What you'll see:**
- Configuration summary (dataset name, column count)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs confirmed (✓)
- DataSource created (✓)
- Progress tracker ready (✓)

**Note:** This operation automatically analyzes all columns in the dataset - no manual field selection required

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Get config
kwargs = create_config_kwargs()

print(f"\n🔍 Configuration...\n")
print(f"✓ Dataset name: '{kwargs['dataset_name']}'")
print(f"  Total columns to analyze: {len(df.columns)}")
print(f"  Columns: {', '.join(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="profiler_001",
    task_type="attribute_profiling",
    description="Automatic attribute profiling and categorization of dataset columns.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs for execute
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Profiling dataset attributes",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review operation configuration
- Run to execute attribute profiling
- Monitor progress bar (6 processing steps)

**What you'll see:**
- Configuration summary (language, sample size, settings)
- Progress bar: validate → profile → categorize → calculate metrics → visualize → save
- Completion message with status and artifact count (4-6 files expected)
- Status = "success" (verify no errors)

**Key parameters:**
- `language='en'`: English keyword matching for categorization
- `sample_size=10`: Sample values per column for analysis
- `max_columns=None`: Analyze all columns (no limit)
- `generate_visualization=True`: Create role distribution charts
- `save_output=True`: Save all results to files

In [ ]:
# Create operation with production-style configuration
operation = DataAttributeProfilerOperation(
    # Core parameters
    name='DataAttributeProfiler',
    language='en',                       # English keyword matching
    sample_size=10,                      # Sample values per column
    max_columns=None,                    # Analyze all columns
    
    # Dictionary configuration
    dictionary_path=None,                # Use default dictionary
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_dask=False,
    chunk_size=10000,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Name:             {operation.name}")
print(f"  Language:         {operation.language}")
print(f"  Sample size:      {operation.sample_size}")
print(f"  Max columns:      {operation.max_columns or 'All'}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing attribute profiling...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and display attribute profiling results
- Review role categorization and privacy metrics
- Check risk scores and entropy values

**What you'll see:**
- Generated artifacts list (roles JSON, entropy CSV, samples)
- Attribute role summary (counts per category)
- Columns grouped by role with confidence scores
- Key metrics (total columns, identifier counts, entropy)
- Top 10 columns by privacy risk (entropy × uniqueness)

**Note:** Higher entropy and uniqueness = higher re-identification risk. Direct identifiers (email, patient_id) pose highest privacy risk.

**Generated artifacts:**
- Attribute roles JSON in output/
- Entropy CSV in output/
- Sample values JSON in output/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load attribute roles file
roles_files = list(task_dir.glob('output/attribute_roles_*.json'))
if roles_files:
    roles_file = roles_files[0]
    
    with open(roles_file, 'r') as f:
        roles_data = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 ATTRIBUTE ANALYSIS RESULTS")
    print("=" * 80)
    
    # Display summary
    summary = roles_data.get('summary', {})
    print("\n🔍 Attribute Role Summary:")
    print("-" * 60)
    for role, count in summary.items():
        role_name = role.replace('_', ' ').title()
        bar = '█' * (count * 5)
        print(f"  {role_name:25s} {bar:20s} {count:2d}")
    
    # Display column groups
    print("\n📋 Columns by Role:")
    print("-" * 60)
    column_groups = roles_data.get('column_groups', {})
    for role, columns in column_groups.items():
        if columns:
            role_name = role.replace('_', ' ').title()
            print(f"\n  {role_name}:")
            for col in columns:
                col_data = roles_data['columns'].get(col, {})
                confidence = col_data.get('confidence', 0)
                print(f"    • {col:20s} (confidence: {confidence:.2f})")
    
    # Display result metrics
    if result.metrics:
        print("\n" + "=" * 80)
        print("📊 KEY METRICS")
        print("=" * 80)
        print(f"  Total columns:          {result.metrics.get('total_columns', 0)}")
        print(f"  Direct identifiers:     {result.metrics.get('direct_identifiers_count', 0)}")
        print(f"  Quasi-identifiers:      {result.metrics.get('quasi_identifiers_count', 0)}")
        print(f"  Sensitive attributes:   {result.metrics.get('sensitive_attributes_count', 0)}")
        print(f"  Indirect identifiers:   {result.metrics.get('indirect_identifiers_count', 0)}")
        print(f"  Non-sensitive:          {result.metrics.get('non_sensitive_count', 0)}")
        
        if 'avg_entropy' in result.metrics:
            print(f"\n  Average entropy:        {result.metrics.get('avg_entropy', 0):.4f}")
            print(f"  Average uniqueness:     {result.metrics.get('avg_uniqueness', 0):.4f}")
    
    print(f"\n💡 Higher entropy and uniqueness = Higher re-identification risk!")
else:
    print("⚠️  No attribute roles file found in", task_dir / 'output')

# Load and display entropy data
entropy_files = list(task_dir.glob('output/attribute_entropy_*.csv'))
if entropy_files:
    entropy_df = pd.read_csv(entropy_files[0])
    
    print("\n" + "=" * 80)
    print("📈 TOP 10 COLUMNS BY RISK (Entropy × Uniqueness)")
    print("=" * 80)
    
    # Calculate risk score
    entropy_df['risk_score'] = entropy_df['entropy'] * entropy_df['uniqueness_ratio']
    entropy_df = entropy_df.sort_values('risk_score', ascending=False)
    
    print(f"\n{'Column':<20} {'Role':<25} {'Entropy':>8} {'Unique':>8} {'Risk':>8}")
    print("-" * 80)
    for _, row in entropy_df.head(10).iterrows():
        role_short = row['role'].replace('_', ' ').title()[:24]
        print(f"{row['column_name']:<20} {role_short:<25} {row['entropy']:>8.2f} {row['uniqueness_ratio']:>8.2%} {row['risk_score']:>8.2f}")

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files in directory structure
- Navigate to task_dir for manual inspection
- Verify file counts and sizes

**What you'll see:**
- Directory structure with file counts per folder
- File names with sizes (KB) for up to 5 files per folder
- Full path to task directory for external access
- Total artifacts confirmation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Analysis JSON and CSV files
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Attribute Roles**: Detailed categorization of each column
3. **Entropy Analysis**: Statistical properties of columns
4. **Sample Values**: Representative values from each column
5. **Visualizations**: Charts showing attribute distribution and characteristics

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display attribute counts
            print("\n📊 ATTRIBUTE ROLE COUNTS:")
            print(f"   Total Columns: {metrics.get('total_columns', 'N/A')}")
            print(f"   Direct Identifiers: {metrics.get('direct_identifiers_count', 'N/A')}")
            print(f"   Quasi-Identifiers: {metrics.get('quasi_identifiers_count', 'N/A')}")
            print(f"   Sensitive Attributes: {metrics.get('sensitive_attributes_count', 'N/A')}")
            print(f"   Indirect Identifiers: {metrics.get('indirect_identifiers_count', 'N/A')}")
            print(f"   Non-Sensitive: {metrics.get('non_sensitive_count', 'N/A')}")
            
            # Display dataset metrics
            if 'avg_entropy' in metrics:
                print("\n📈 DATASET METRICS:")
                print(f"   Average Entropy: {metrics.get('avg_entropy', 0):.4f}")
                print(f"   Average Uniqueness: {metrics.get('avg_uniqueness', 0):.4f}")
            
            # Display quasi-identifiers if present
            if 'quasi_identifiers' in metrics:
                qi_list = metrics['quasi_identifiers']
                if qi_list:
                    print("\n🔍 QUASI-IDENTIFIERS:")
                    for qi in qi_list[:10]:
                        print(f"   • {qi}")
                    if len(qi_list) > 10:
                        print(f"   ... and {len(qi_list) - 10} more")
            
            # Display conflicts if any
            if 'conflicts_count' in metrics and metrics['conflicts_count'] > 0:
                print("\n⚠️  CONFLICTS DETECTED:")
                print(f"   Count: {metrics.get('conflicts_count', 0)}")
                print("   (See attribute_roles file for details)")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. ATTRIBUTE ROLES ANALYSIS (NEWEST)
print("\n\n2️⃣ ATTRIBUTE ROLES ANALYSIS:")
print("=" * 80)

output_dir = task_dir / "output"
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    roles_files = sorted(
        output_dir.glob("attribute_roles_*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not roles_files:
        print("⚠️  No attribute roles files found")
    else:
        latest_roles_file = roles_files[0]
        mtime = datetime.fromtimestamp(latest_roles_file.stat().st_mtime)

        print(f"📄 File      : {latest_roles_file.name}")
        print(f"🕒 Modified  : {mtime:%Y-%m-%d %H:%M:%S}")
        print(f"📦 Size      : {latest_roles_file.stat().st_size / 1024:.1f} KB\n")

        try:
            with open(latest_roles_file, "r", encoding="utf-8") as f:
                roles_data = json.load(f)

            # DATASET OVERVIEW
            dataset_info = roles_data.get("dataset_info", {})
            summary = roles_data.get("summary", {})
            dataset_metrics = roles_data.get("dataset_metrics", {})

            print("📊 DATASET OVERVIEW")
            print("-" * 80)
            print(
                f"Rows: {dataset_info.get('rows', 'N/A'):,} | "
                f"Columns: {dataset_info.get('columns', 'N/A')} | "
                f"Analyzed at: {dataset_info.get('analyzed_at', 'N/A')}"
            )

            print(
                f"Avg entropy: {dataset_metrics.get('avg_entropy', 0):.2f} | "
                f"Avg uniqueness: {dataset_metrics.get('avg_uniqueness', 0):.2%}\n"
            )

            print("📌 ROLE SUMMARY")
            for role, count in summary.items():
                print(f"  • {role:<20}: {count}")
            print()

            # COLUMNS BY ROLE
            print("📋 COLUMNS BY ROLE (detailed)")
            print("-" * 80)

            column_groups = roles_data.get("column_groups", {})
            columns_data = roles_data.get("columns", {})

            role_order = [
                "DIRECT_IDENTIFIER",
                "QUASI_IDENTIFIER",
                "SENSITIVE_ATTRIBUTE",
                "INDIRECT_IDENTIFIER",
                "NON_SENSITIVE",
            ]

            for role in role_order:
                cols = column_groups.get(role, [])
                if not cols:
                    continue

                print(f"\n🔹 {role.replace('_', ' ').title()} ({len(cols)} columns)")
                print("-" * 80)

                for col in cols:
                    col_data = columns_data.get(col, {})
                    stats = col_data.get("statistics", {})

                    print(
                        f"• {col}\n"
                        f"    role        : {col_data.get('role')} "
                        f"(conf={col_data.get('confidence', 0):.2f})\n"
                        f"    type        : {stats.get('data_type')} / {stats.get('inferred_type')}\n"
                        f"    missing     : {stats.get('missing_rate', 0):.2%}\n"
                        f"    entropy     : {stats.get('entropy', 0):.2f} "
                        f"(norm={stats.get('normalized_entropy', 0):.2f})\n"
                        f"    uniqueness  : {stats.get('uniqueness_ratio', 0):.2%}\n"
                    )

            # CONFLICTS
            conflicts = roles_data.get("conflicts", [])
            if conflicts:
                print("\n⚠️  CATEGORIZATION CONFLICTS")
                print("-" * 80)

                for conflict in conflicts:
                    col = conflict.get("column")
                    details = conflict.get("details", {})

                    print(
                        f"• {col}\n"
                        f"    semantic     : {details.get('semantic_category')} "
                        f"(conf={details.get('semantic_confidence', 0):.2f})\n"
                        f"    statistical  : {details.get('statistical_category')} "
                        f"(conf={details.get('statistical_confidence', 0):.2f})\n"
                    )

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading attribute roles: {e}")

# 3. ENTROPY ANALYSIS (NEWEST)
print("\n\n3️⃣ ENTROPY & UNIQUENESS ANALYSIS:")
print("=" * 80)

entropy_files = sorted(
    output_dir.glob("attribute_entropy_*.csv"),
    key=lambda x: x.stat().st_mtime,
    reverse=True,
)

if not entropy_files:
    print("⚠️  No entropy files found")
else:
    latest_entropy_file = entropy_files[0]
    mtime = datetime.fromtimestamp(latest_entropy_file.stat().st_mtime)

    print(f"📄 File      : {latest_entropy_file.name}")
    print(f"🕒 Modified  : {mtime:%Y-%m-%d %H:%M:%S}")
    print(f"📦 Size      : {latest_entropy_file.stat().st_size / 1024:.1f} KB\n")

    try:
        df = pd.read_csv(latest_entropy_file)

        # SANITY & DERIVED METRICS
        df["entropy"] = df["entropy"].clip(lower=0)
        df["risk_score"] = df["entropy"] * df["uniqueness_ratio"]

        df = df.sort_values("risk_score", ascending=False)

        # TOP HIGH-RISK COLUMNS
        print("🔥 TOP 15 COLUMNS BY PRIVACY RISK\n")
        print(
            f"{'Column':<25} {'Role':<22} {'Type':<10} "
            f"{'Entropy':>8} {'Unique':>8} {'Risk':>8}"
        )
        print("-" * 95)

        for _, row in df.head(15).iterrows():
            print(
                f"{row['column_name']:<25} "
                f"{row['role'].replace('_',' ').title():<22} "
                f"{row['inferred_type']:<10} "
                f"{row['entropy']:>8.2f} "
                f"{row['uniqueness_ratio']:>8.2%} "
                f"{row['risk_score']:>8.2f}"
            )

        # ROLE-BASED RISK SUMMARY
        print("\n📊 RISK SUMMARY BY ROLE")
        print("-" * 80)

        role_summary = (
            df.groupby("role")
            .agg(
                columns=("column_name", "count"),
                avg_entropy=("entropy", "mean"),
                avg_uniqueness=("uniqueness_ratio", "mean"),
                avg_risk=("risk_score", "mean"),
                max_risk=("risk_score", "max"),
            )
            .sort_values("avg_risk", ascending=False)
        )

        for role, r in role_summary.iterrows():
            print(
                f"• {role:<20} | "
                f"cols={int(r.columns):<2} | "
                f"avg_entropy={r.avg_entropy:>5.2f} | "
                f"avg_unique={r.avg_uniqueness:>6.2%} | "
                f"avg_risk={r.avg_risk:>6.2f} | "
                f"max_risk={r.max_risk:>6.2f}"
            )

        # GLOBAL STATS
        print("\n📈 GLOBAL STATISTICS")
        print("-" * 80)

        print(f"Average entropy        : {df['entropy'].mean():.4f}")
        print(f"Average uniqueness     : {df['uniqueness_ratio'].mean():.4f}")
        print(f"Average risk score     : {df['risk_score'].mean():.4f}")
        print(f"> 90% uniqueness cols  : {(df['uniqueness_ratio'] > 0.9).sum()}")
        print(f"> 8.0 entropy cols     : {(df['entropy'] > 8.0).sum()}")
        print(f"> 1.0 risk score cols  : {(df['risk_score'] > 1.0).sum()}")

        # OPTIONAL: HIGH-RISK SENSITIVE CHECK
        high_sensitive = df[
            (df["role"] == "SENSITIVE_ATTRIBUTE") & (df["risk_score"] > 1.0)
        ]

        if not high_sensitive.empty:
            print("\n🚨 HIGH-RISK SENSITIVE ATTRIBUTES")
            print("-" * 80)
            for _, r in high_sensitive.iterrows():
                print(
                    f"• {r['column_name']} "
                    f"(entropy={r['entropy']:.2f}, "
                    f"unique={r['uniqueness_ratio']:.2%}, "
                    f"risk={r['risk_score']:.2f})"
                )

    except Exception as e:
        print(f"❌ Error reading entropy data: {e}")

# 4. SAMPLE VALUES (NEWEST)
print("\n\n4️⃣ SAMPLE VALUES (PRIVACY-SAFE PREVIEW):")
print("=" * 80)

sample_files = sorted(
    output_dir.glob("attribute_sample_*.json"),
    key=lambda x: x.stat().st_mtime,
    reverse=True,
)

if not sample_files:
    print("⚠️  No sample values files found")
else:
    latest_sample_file = sample_files[0]
    mtime = datetime.fromtimestamp(latest_sample_file.stat().st_mtime)

    print(f"📄 File     : {latest_sample_file.name}")
    print(f"🕒 Modified : {mtime:%Y-%m-%d %H:%M:%S}")
    print(f"📦 Size     : {latest_sample_file.stat().st_size / 1024:.1f} KB\n")

    try:
        with open(latest_sample_file, "r", encoding="utf-8") as f:
            sample_data = json.load(f)

        def mask_value(val, role):
            if val is None:
                return None

            if role == "DIRECT_IDENTIFIER":
                s = str(val)
                if "@" in s:  # email
                    return s[0] + "***@" + s.split("@")[1].split(".")[0] + ".***"
                if s.startswith("+"):
                    return s[:3] + "-***-***-" + s[-2:]
                return "***"

            if role == "SENSITIVE_ATTRIBUTE":
                if isinstance(val, (int, float)):
                    return f"~{round(val, -3):,.0f}"
                return "***"

            return val

        # SHOW ONLY HIGH-RISK / INTERESTING COLUMNS
        print("📋 SAMPLE PREVIEW BY COLUMN (TOP 10 RISK RELEVANT)\n")

        shown = 0
        for col_name, col_info in sample_data.items():
            role = col_info.get("role", "UNKNOWN")
            inferred_type = col_info.get("inferred_type", "unknown")
            samples = col_info.get("samples", [])

            # Skip boring columns
            if role == "NON_SENSITIVE" and inferred_type == "categorical":
                continue

            shown += 1
            safe_samples = [mask_value(v, role) for v in samples[:5]]

            print(f"{shown}. {col_name}")
            print(f"   Role : {role.replace('_', ' ').title()}")
            print(f"   Type : {inferred_type}")
            print(f"   Preview : {safe_samples}")

            if role in {"DIRECT_IDENTIFIER", "SENSITIVE_ATTRIBUTE"}:
                print("   ⚠️  High-risk attribute – masking applied")

            print()

            if shown >= 10:
                break

        remaining = len(sample_data) - shown
        if remaining > 0:
            print(f"... ({remaining} more columns omitted for brevity & safety)")

    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
    except Exception as e:
        print(f"❌ Error reading sample values: {e}")

# 5. VISUALIZATIONS (NEWEST BATCH)
print("\n\n5️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review attribute roles and privacy metrics  

**Key takeaways:**
- Automatic attribute role detection (identifiers, quasi-identifiers, sensitive)
- Entropy and uniqueness metrics indicate re-identification risk
- Direct identifiers = highest risk (email, patient_id, names)
- Quasi-identifiers = combinable attributes (birth_date, gender, region)
- Sensitive attributes = confidential data (diagnosis, salary, ethnicity)
- All analysis saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_data_attribute_profiler_advanced.ipynb`** includes:
- Custom attribute dictionaries
- Multi-language support
- Conflict resolution strategies
- Advanced statistical analysis
- Integration with anonymization pipelines
- Batch processing for multiple datasets

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Privacy Metrics Guide](../../docs/privacy_metrics.md)
- 📋 [Attribute Dictionary Format](../../docs/attribute_dictionary.md)